# 📊 Tech Challenge: Voice of Customer (VoC) Intelligence com RAG
Bem-vindo ao projeto de exploração inteligente de avaliações do e-commerce brasileiro (base Olist).

**O Problema de Negócio:**
Empresas recebem milhares de avaliações diariamente. Analisar manualmente comentários textuais para extrair insights (motivos de insatisfação, padrões de elogios) é uma tarefa lenta e pouco escalável.

**A Solução:**
Vamos construir uma arquitetura **RAG (Retrieval-Augmented Generation)**. Transformaremos as avaliações em uma base de conhecimento inteligente, permitindo que gestores façam perguntas em linguagem natural e recebam análises precisas, fundamentadas em evidências reais (comentários, notas e categorias).

---
## 🛠️ Fase 1: Aquisição, Limpeza e Enriquecimento de Dados
Nesta primeira etapa, vamos carregar os dados brutos e prepará-los para a nossa Inteligência Artificial.

In [ ]:
import warnings
warnings.filterwarnings('ignore') # Ignora os avisos visuais que poluem o notebook

import pandas as pd
import re

### 1.1 Carregamento dos Dados Brutos
Vamos carregar as bases originais do Olist. O foco principal é o dataset de *Reviews*.

In [ ]:
# Carregando as bases de dados principais
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')
df_items = pd.read_csv('olist_order_items_dataset.csv')
df_products = pd.read_csv('olist_products_dataset.csv')

print(f"Total de avaliações originais: {len(df_reviews)}")

### 1.2 Filtragem e Cruzamento (Joins)
**Estratégia para o Desafio Adicional:**
Para o nosso sistema RAG, avaliações apenas com a nota (sem texto) não possuem valor semântico. Vamos filtrar apenas os clientes que deixaram um comentário.
Além disso, para permitir buscas por **Categoria do Produto** e **Data**, precisamos cruzar a tabela de *reviews* com as tabelas de *orders*, *items* e *products*.

In [ ]:
# 1. Manter apenas avaliações que possuam comentário em texto
df_base = df_reviews.dropna(subset=['review_comment_message']).copy()
print(f"Avaliações com texto disponíveis: {len(df_base)}")

# 2. Trazer a data do pedido (para análises temporais no RAG)
df_base = df_base.merge(df_orders[['order_id', 'order_purchase_timestamp']], on='order_id', how='left')

# 3. Trazer o ID do produto
# Um pedido pode ter vários itens iguais, usamos drop_duplicates para não duplicar a avaliação
df_items_unique = df_items[['order_id', 'product_id']].drop_duplicates(subset=['order_id'])
df_base = df_base.merge(df_items_unique, on='order_id', how='left')

# 4. Trazer a categoria do produto (Metadata bônus do desafio)
df_base = df_base.merge(df_products[['product_id', 'product_category_name']], on='product_id', how='left')

# Preencher categorias vazias com 'desconhecido'
df_base['product_category_name'] = df_base['product_category_name'].fillna('desconhecido')

df_base.head(3)

### 1.3 Limpeza e Preparação dos Textos
Textos extraídos da web costumam vir com quebras de linha (`\n`) e espaços múltiplos, o que prejudica a qualidade dos *embeddings*. 
Vamos aplicar uma função de limpeza para padronizar o conteúdo e unir o título da avaliação ao corpo da mensagem, formando um contexto rico para a IA.

In [ ]:
def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto)
    # Remove quebras de linha
    texto = re.sub(r'[\r\n]+', ' ', texto)
    # Remove espaços duplos
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

# Aplicar limpeza no título e na mensagem
df_base['review_comment_title'] = df_base['review_comment_title'].apply(limpar_texto)
df_base['review_comment_message'] = df_base['review_comment_message'].apply(limpar_texto)

# Criar a coluna final de texto (Contexto) que será vetorizada
# Se tiver título, junta com a mensagem. Se não, usa só a mensagem.
df_base['texto_completo'] = df_base.apply(
    lambda row: f"{row['review_comment_title']} - {row['review_comment_message']}" if row['review_comment_title'] != "" else row['review_comment_message'],
    axis=1
)

# Converter a data para formato datetime e extrair apenas ano/mês
df_base['order_purchase_timestamp'] = pd.to_datetime(df_base['order_purchase_timestamp'])
df_base['ano_mes'] = df_base['order_purchase_timestamp'].dt.strftime('%Y-%m')

df_base[['review_score', 'product_category_name', 'texto_completo']].head()

### 1.4 Seleção Final e Exportação
Agora, vamos reter apenas os metadados necessários e remover comentários extremamente curtos (ex: "ok", "bom"), pois não agregam profundidade analítica. Salvamos a base limpa para não precisarmos reprocessar tudo novamente.

In [ ]:
# Selecionar apenas as colunas relevantes para o banco vetorial
colunas_finais = [
    'review_id', 
    'order_id', 
    'review_score', 
    'product_category_name', 
    'ano_mes', 
    'texto_completo'
]

df_voce_cliente = df_base[colunas_finais].copy()

# Remover textos muito curtos (limite mínimo de 15 caracteres)
df_voce_cliente = df_voce_cliente[df_voce_cliente['texto_completo'].str.len() > 15]

print(f"Base final limpa e pronta para embeddings: {len(df_voce_cliente)} avaliações")

# Exportar base
df_voce_cliente.to_csv('base_conhecimento_limpa.csv', index=False)

---
## 🧠 Fase 2: Criação dos Embeddings e do Banco Vetorial
Nesta etapa, transformaremos cada avaliação em um **vetor matemático** (Embeddings) utilizando o modelo `paraphrase-multilingual-MiniLM-L12-v2`. Isso permite que o sistema compreenda o *significado* da frase, mesmo se as palavras exatas não forem usadas na pergunta.
Armazenaremos esses vetores em um banco de dados focado em IA: o **ChromaDB**.

In [ ]:
# Instalação silenciosa (-q) das bibliotecas de IA
!pip install langchain langchain-community langchain-huggingface sentence-transformers chromadb tqdm -q

In [ ]:
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from tqdm import tqdm

In [ ]:
# 1. Carregar a base limpa
df_voce_cliente = pd.read_csv('base_conhecimento_limpa.csv')

# 2. Transformar o DataFrame em objetos Document do LangChain
documentos = []
for _, row in tqdm(df_voce_cliente.iterrows(), total=len(df_voce_cliente), desc="Preparando documentos"):
    doc = Document(
        page_content=str(row['texto_completo']),
        metadata={
            "review_id": str(row['review_id']),
            "order_id": str(row['order_id']),
            "review_score": int(row['review_score']),
            "product_category_name": str(row['product_category_name']),
            "ano_mes": str(row['ano_mes'])
        }
    )
    documentos.append(doc)

# 3. Inicializar o modelo de Embeddings (Multilíngue otimizado para semântica em Português)
print("\nCarregando o modelo de embeddings...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 4. Criar e salvar o Banco Vetorial Localmente
print("Criando o ChromaDB...")
vectordb = Chroma.from_documents(
    documents=documentos, 
    embedding=embeddings_model, 
    persist_directory="./chroma_db_olist"
)
print("\nSucesso! Banco vetorial indexado na pasta 'chroma_db_olist'.")

### 2.1 Testando o Motor de Busca (Retriever)
Antes de injetarmos um LLM, vamos garantir que a nossa busca vetorial já está pescando as evidências corretas baseada puramente em similaridade semântica.

In [ ]:
# Buscando os 3 resultados mais relevantes
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

pergunta_teste = "O que os clientes estão falando sobre atraso na entrega?"
resultados = retriever.invoke(pergunta_teste)

print(f"Pergunta: {pergunta_teste}\n")
for i, res in enumerate(resultados, 1):
    print(f"--- Evidência {i} ---")
    print(f"Texto: {res.page_content}")
    print(f"Nota: {res.metadata['review_score']} | Categoria: {res.metadata['product_category_name']}")
    print("-" * 20)

---
## 🤖 Fase 3: Orquestração do Pipeline RAG
Aqui conectamos o motor de busca (ChromaDB) ao **Google Gemini 3.5 Flash** usando a orquestração do LangChain.

**Trava Anti-Alucinação:** 
Para atender às exigências do Tech Challenge, o modelo recebe um *Prompt* estrito de VoC (Voice of Customer), obrigando-o a citar as evidências e a admitir limitação caso uma pergunta fuja do contexto da base (ex: receitas de bolo).

In [ ]:
# Instalação silenciosa das bibliotecas do Google
!pip install langchain-google-genai google-generativeai python-dotenv -q

In [ ]:
import os
import google.generativeai as genai
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Carrega as variáveis do arquivo .env (Onde a chave da API está escondida de forma segura)
load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    print("✅ Chave de API carregada com sucesso!")
    genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
else:
    print("❌ ERRO: A variável GOOGLE_API_KEY não foi encontrada. Verifique seu arquivo .env.")

In [ ]:
# 1. Instanciar o LLM (Gemini 3.5 Flash: rápido, poderoso e otimizado para RAG)
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)

# 2. Criar o Prompt System com as amarras de segurança exigidas no desafio
template_voc = """
Você é um analista de Voice of Customer (VoC) de um e-commerce.
Seu objetivo é responder à pergunta do gestor utilizando estritamente o contexto de avaliações fornecido.

Regras de negócio:
1. Responda baseando-se APENAS no contexto abaixo.
2. Se o contexto não contiver informações suficientes ou a pergunta fugir do escopo, responda EXATAMENTE: "Não possuo evidências suficientes na base de avaliações para responder a esta pergunta."
3. Sempre cite as evidências: mencione trechos dos comentários e faça referência à Categoria do produto e à Nota atribuída para validar seu ponto.

Contexto (Avaliações recuperadas):
{context}

Pergunta do gestor: {question}

Resposta analítica:
"""
prompt = PromptTemplate.from_template(template_voc)

# 3. Função para injetar os metadados diretamente no texto que a IA vai ler
def formatar_docs_com_metadados(docs):
    contextos_formatados = []
    for d in docs:
        nota = d.metadata.get('review_score', 'N/A')
        categoria = d.metadata.get('product_category_name', 'N/A')
        texto = d.page_content
        contextos_formatados.append(f"[Nota: {nota} | Categoria: {categoria}] Comentário: {texto}")
    return "\n\n".join(contextos_formatados)

# 4. Construir a Chain (Fluxo de Execução do LangChain)
rag_chain = (
    {"context": retriever | formatar_docs_com_metadados, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 3.1 Avaliando o RAG e o Filtro de Segurança
Vamos fazer duas perguntas. Uma real de negócios e outra completamente fora de escopo para verificar se a IA resiste a alucinar informações.

In [ ]:
print("TESTE 1: Pergunta de Negócio Válida")
pergunta_valida = "Quais problemas os clientes relatam sobre a entrega?"
print(f"\n{rag_chain.invoke(pergunta_valida)}\n")

print("-" * 50)

print("\nTESTE 2: Pergunta Fora do Escopo (Tratamento de Alucinação)")
pergunta_fora_escopo = "Qual a receita para fazer um bolo de chocolate?"
print(f"\n{rag_chain.invoke(pergunta_fora_escopo)}")

---
## 💻 Fase 4: O Código da Aplicação Interativa (Streamlit)
Para o trabalho ter cara de produto final de mercado, vamos empacotar o código acima em um painel interativo. 

**Atenção:** O comando mágico `%%writefile app.py` abaixo escreve o código no disco em vez de tentar executá-lo no Jupyter. Isso **resolve o problema de lentidão e os avisos de `missing ScriptRunContext`** que poluíam o ambiente.

Após rodar a célula abaixo, abra o terminal do Anaconda e digite:
`streamlit run app.py`

In [ ]:
%%writefile app.py
import streamlit as st
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Configuração da Página do Streamlit
st.set_page_config(page_title="VoC Intelligence", page_icon="📊", layout="centered")
st.title("📊 Voice of Customer Intelligence")
st.markdown("Exploração inteligente de avaliações do e-commerce utilizando Retrieval-Augmented Generation (RAG) e Filtros de Metadados.")

# 2. Carregar variáveis de ambiente
load_dotenv()
if "GOOGLE_API_KEY" not in os.environ:
    st.error("Erro: GOOGLE_API_KEY não encontrada. Verifique o arquivo .env na mesma pasta do app.py.")
    st.stop()

# 3. Funções Lazy (Carregamento Sob Demanda)
def carregar_banco_vetorial(categoria_filtro=None):
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import Chroma
    
    # Descobre a pasta exata onde o app.py está rodando, evitando erros de Path
    diretorio_atual = os.path.dirname(os.path.abspath(__file__))
    caminho_banco = os.path.join(diretorio_atual, "chroma_db_olist")
    
    embeddings_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    vectordb = Chroma(persist_directory=caminho_banco, embedding_function=embeddings_model)
    
    search_kwargs = {"k": 7}
    if categoria_filtro and categoria_filtro != "Todas as Categorias":
        search_kwargs["filter"] = {"product_category_name": categoria_filtro}
        
    return vectordb.as_retriever(search_kwargs=search_kwargs)

def formatar_docs_com_metadados(docs):
    contextos = []
    for d in docs:
        nota = d.metadata.get('review_score', 'N/A')
        categoria = d.metadata.get('product_category_name', 'N/A')
        texto = d.page_content
        contextos.append(f"[Nota: {nota} | Categoria: {categoria}] Comentário: {texto}")
    return "\n\n".join(contextos)

def montar_pipeline(retriever):
    llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)
    
    template_voc = """
    Você é um analista de Voice of Customer (VoC) de um e-commerce.
    Seu objetivo é responder à pergunta do gestor utilizando estritamente o contexto de avaliações fornecido.

    Regras de negócio:
    1. Responda baseando-se APENAS no contexto abaixo.
    2. Se o contexto não contiver informações suficientes ou a pergunta fugir do escopo, responda EXATAMENTE: "Não possuo evidências suficientes na base de avaliações para responder a esta pergunta."
    3. Sempre cite as evidências: mencione trechos dos comentários e faça referência à Categoria do produto e à Nota atribuída para validar seu ponto.
    4. Estruture a resposta em tópicos para facilitar a leitura.

    Contexto (Avaliações recuperadas):
    {context}

    Pergunta do gestor: {question}

    Resposta analítica:
    """
    prompt = PromptTemplate.from_template(template_voc)
    
    rag_chain = (
        {"context": retriever | formatar_docs_com_metadados, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain

# 4. Interface de Interação
col1, col2 = st.columns([2, 1])
with col1:
    pergunta_usuario = st.text_input(
        "O que você deseja saber sobre a opinião dos clientes?", 
        placeholder="Ex: Quais são as principais reclamações sobre a qualidade?"
    )
with col2:
    lista_categorias = [
        "Todas as Categorias", "beleza_saude", "cama_mesa_banho", 
        "esporte_lazer", "informatica_acessorios", "moveis_decoracao", 
        "utilidades_domesticas", "relogios_presentes", "brinquedos"
    ]
    categoria_selecionada = st.selectbox("Filtrar por Categoria:", lista_categorias)

if st.button("Analisar Sentimento e Avaliações"):
    if pergunta_usuario:
        with st.spinner("Conectando ao banco de avaliações e gerando análise... (isso pode levar alguns segundos)"):
            try:
                retriever = carregar_banco_vetorial(categoria_selecionada)
                rag_chain = montar_pipeline(retriever)
                
                resposta = rag_chain.invoke(pergunta_usuario)
                
                st.markdown("### 📋 Análise Gerencial")
                st.write(resposta)
                
                with st.expander("🔍 Ver Evidências Recuperadas (Contexto Bruto)"):
                    docs_recuperados = retriever.invoke(pergunta_usuario)
                    for i, doc in enumerate(docs_recuperados, 1):
                        st.markdown(f"**Evidência {i}** (Nota: {doc.metadata.get('review_score')} | Categoria: {doc.metadata.get('product_category_name')})")
                        st.write(f"_{doc.page_content}_")
                        st.divider()
                        
            except Exception as e:
                st.error(f"Erro ao processar a pergunta: {e}")
    else:
        st.warning("Por favor, digite uma pergunta antes de solicitar a análise.")